# Specter Mutual: zero to a deployed app

**Time:** 40 minutes, in four parts

**What you build:** an AI claims reviewer for Specter Mutual, a fictional insurer that covers paranormal damage. A RocketRide **agent** reads each claim and recommends approve, deny or escalate. A **policy engine**, plain code with no AI in it, holds that recommendation to the rules. A **claims portal** puts it in front of people, and you deploy it to your RocketRide workspace.

**Done when:** the last part prints **Open your portal**, and the link shows Specter Mutual reviewing a claim you filed.

The agent recommends and the engine decides: an agent can be talked into things, plain code can't.

| Part | Notebook | Time | You leave with |
| --- | --- | --- | --- |
| 1 | [Project init](01-project-init.ipynb) | 3 min | A new RocketRide app, created from nothing |
| 2 | [Pipeline build](02-pipeline-build.ipynb) | 15 min | An agent that decides, and a policy engine that holds it to the rules |
| 3 | [UI build](03-ui-build.ipynb) | 14 min | The claims portal, built on your laptop |
| 4 | [Cloud deploy](04-cloud-deploy.ipynb) | 5 min | The portal live in your workspace |

> Run this notebook's cells once to check your machine (**Run All**). Then open the parts in order. Each part loads this notebook first with `%run`, so its settings and helpers are shared, and everything a part makes is saved under `build/` for the next one.
> You need Python 3.10+, Node 20+ and pnpm 10+, a RocketRide API key, a developer id for your organization (Part 1 explains), and `ROCKETRIDE_ANTHROPIC_KEY` in your RocketRide Variables (the portal can save it for you).

## Preflight

Installs the RocketRide Python SDK into this notebook's Python if it's missing, then checks for Node and pnpm. Fix anything marked ❌, then run it again.

In [ ]:
# Preflight: installs the RocketRide SDK, checks Node and pnpm. Prints "ready" when all pass.
import importlib, re, shutil, site, subprocess, sys
from importlib import metadata

if sys.version_info < (3, 10):
    raise RuntimeError(f"This notebook needs Python 3.10 or newer, and this kernel is {sys.version.split()[0]}. "
                       "Pick a newer Python with Select Kernel in the top right.")

SDK, MIN_VERSION = "rocketride>=1.4.0", (1, 4, 0)

def sdk_version():
    try:
        return tuple(int(n) for n in re.findall(r"\d+", metadata.version("rocketride"))[:3])
    except metadata.PackageNotFoundError:
        return None

def pip_install(*extra):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", SDK, *extra],
                          capture_output=True, text=True)

if (sdk_version() or (0,)) < MIN_VERSION:
    print("Installing the RocketRide SDK. This takes about a minute, once.")
    result = pip_install()
    if result.returncode and "externally-managed" in result.stderr:
        # Homebrew and Linux system Pythons block pip; install for this user instead.
        result = pip_install("--user", "--break-system-packages")
        site.addsitedir(site.getusersitepackages())
    if result.returncode:
        raise RuntimeError(f"Couldn't install the RocketRide SDK:\n{result.stderr}")
    importlib.invalidate_caches()
    if "rocketride" in sys.modules:
        raise RuntimeError("Updated the RocketRide SDK. Restart the kernel, then run the notebook again.")

def tool_version(name):
    path = shutil.which(name)
    if not path:
        return None
    out = subprocess.run([path, "--version"], capture_output=True, text=True, timeout=30).stdout
    m = re.search(r"(\d+)\.(\d+)", out)
    return (int(m[1]), int(m[2])) if m else (0, 0)

problems = []
for name, need, fix in [
    ("node", (20, 0), "install Node.js 20 or newer from nodejs.org"),
    ("pnpm", (10, 0), "run  npm install -g pnpm@latest"),
]:
    have = tool_version(name)
    if have is None:
        problems.append(f"{name} is not installed or not on your PATH. Fix: {fix}, then restart VS Code.")
    elif have < need:
        problems.append(f"{name} {have[0]}.{have[1]} is too old (need {need[0]}+). Fix: {fix}.")
if problems:
    for p in problems:
        print("❌", p)
    raise RuntimeError(f"Preflight failed: {len(problems)} problem(s). Fix them, then run this cell again.")
print("\033[32m✅ ready\033[0m")

## Config

Reads your RocketRide settings from the environment or a `.env` file next to this notebook. If your API key isn't set, it asks for it. Nothing is saved.

In [ ]:
# Config: the ONLY cell that reads environment variables. Every later cell uses CONFIG.
import getpass, os
from pathlib import Path

def read_dotenv(path=Path(".env")):
    values = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            key, sep, value = line.strip().partition("=")
            if sep and key.strip().startswith("ROCKETRIDE_"):
                values[key.strip()] = value.strip().strip("\"'")
    return values

env = read_dotenv()  # a .env file next to this notebook...
env.update({k: v for k, v in os.environ.items() if k.startswith("ROCKETRIDE_")})  # ...env wins
env.setdefault("ROCKETRIDE_URI", "https://api.rocketride.ai")
if not env.get("ROCKETRIDE_APIKEY"):
    previous = globals().get("CONFIG", {}).get("apikey")  # don't re-prompt on re-run
    env["ROCKETRIDE_APIKEY"] = previous or getpass.getpass("RocketRide API key: ").strip()

CONFIG = {
    "uri": env["ROCKETRIDE_URI"],
    "apikey": env["ROCKETRIDE_APIKEY"],
    "env": env,
}
print("Server: ", CONFIG["uri"])
print("API key:", "set" if CONFIG["apikey"] else "MISSING")

## Shared helpers

One RocketRide client, the paths every part uses, and three helpers. This cell doesn't touch the network, so `%run` can load it: each part connects with `await connect()` right after.

In [ ]:
# Shared: the client, paths and helpers for every part. Safe to re-run.
import json, shutil, subprocess
from pathlib import Path
from rocketride import RocketRideClient

WORKSPACE = Path("build").resolve()            # everything the parts make lives here
APP_ROOT = WORKSPACE / "apps" / "specter-mutual"
PIPELINES = WORKSPACE / "pipelines"

if "client" not in globals():
    client = RocketRideClient(uri=CONFIG["uri"], auth=CONFIG["apikey"], env=CONFIG["env"])

async def connect():
    """Connect the shared client once. Every part runs this right after loading this notebook."""
    if not client.is_connected():
        await client.connect()
    print("Connected to", CONFIG["uri"])

def developer_id():
    """Your organization's developer id: the first part of every app id."""
    org = (client.get_account_info() or {}).get("organization") or {}
    return org.get("developerId")

def app_id():
    return f"{developer_id()}.specter-mutual"

def sh(*args, cwd=None):
    """Run a command and show its output. Raises if it fails."""
    r = subprocess.run([shutil.which(args[0]) or args[0], *args[1:]], cwd=cwd,
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    print((r.stdout + r.stderr).strip()[-2000:])
    r.check_returncode()

def replay(notebook):
    """Re-run a part's file-writing cells (%%writefile, or cells headed "# Write"). Skip-ahead cells use it."""
    nb = json.loads(Path(notebook).read_text(encoding="utf-8"))
    for cell in nb["cells"]:
        src = "".join(cell["source"])
        if cell["cell_type"] != "code":
            continue
        if src.startswith("%%writefile "):
            first, _, body = src.partition("\n")
            path = Path(first.split(None, 1)[1].strip())
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(body, encoding="utf-8", newline="\n")
            print("Wrote", path.as_posix())
        elif src.startswith("# Write"):
            exec(src, globals())

print("Shared helpers ready. Open Part 1: 01-project-init.ipynb")

## What's next

- **Docs:** [docs.rocketride.org](https://docs.rocketride.org)
- **Help and community:** [RocketRide Discord](https://discord.gg/PMXrtenMsY)
- **Back to the hub:** [RocketRide workshops](https://github.com/rocketride-org/rocketride-workshops)